# Lab 93: fine-tune the desk's router

Run this notebook on Google Colab with a **T4 GPU** (Runtime → Change runtime type → T4 GPU).
It teaches Qwen3 1.7B one job, choosing the research route, with a small LoRA adapter,
and exports the result as one GGUF file for Ollama.


## 1. Install Unsloth
The version Colab installs is printed below; note it next to your result.

In [ ]:
!pip install --quiet unsloth

In [ ]:
import unsloth

print("unsloth", unsloth.__version__)

## 2. Upload the training data
Upload `router_train.jsonl` from `edge/finetune/data/` on your computer.

In [ ]:
from google.colab import files

uploaded = files.upload()  # choose router_train.jsonl
print(list(uploaded))

## 3. Load the base model in 4-bit and add a LoRA adapter

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-1.7B",
    max_seq_length=1024,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=93,
)

## 4. Turn every chat into the model's own text format

In [ ]:
from datasets import load_dataset

data = load_dataset("json", data_files="router_train.jsonl", split="train")


def to_text(row):
    text = tokenizer.apply_chat_template(row["messages"], tokenize=False, enable_thinking=False)
    return {"text": text}


data = data.map(to_text)
print(data[0]["text"])

## 5. Train (a few minutes on a T4)

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=data,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        seed=93,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer.train()

## 6. Export one GGUF file for Ollama and download it

In [ ]:
import glob

model.save_pretrained_gguf("router_gguf", tokenizer, quantization_method="q4_k_m")
for path in glob.glob("router_gguf*/**/*.gguf", recursive=True) + glob.glob("*.gguf"):
    print(path)
    files.download(path)